# Practica: Coeficiente de Spearman (2.1.2)

**Asignatura:** Analisis y visualizacion de datos (GAD-2401)  
**Tema:** 2.1.2 – Coeficiente de correlacion de rangos de Spearman (rho)

## Objetivos

1. Calcular el coeficiente de Spearman de forma programatica y entender los rangos.
2. Comparar Spearman con Pearson.
3. Observar la robustez ante outliers.
4. Verificar los ejercicios resueltos a mano en el cuaderno.
5. Aplicar Spearman a un dataset real.

> **Importante:** Resuelve primero los ejercicios de `ejercicios/ejercicios-manuales.html` en tu cuaderno.


## 1. Importacion de librerias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)
print("Librerias cargadas correctamente.")

## 2. Funcion auxiliar: Spearman paso a paso (formula con suma d^2)

Muestra los rangos y el calculo equivalente al del cuaderno.

In [ ]:
def spearman_manual_detalle(x, y, nombre_x="X", nombre_y="Y"):
    """
    Calcula Spearman mostrando rangos y la formula con sum d^2.
    Tambien reporta el valor de scipy (maneja empates).
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)

    rx = stats.rankdata(x, method="average")
    ry = stats.rankdata(y, method="average")
    d = rx - ry
    sum_d2 = np.sum(d**2)

    rho_formula = 1 - (6 * sum_d2) / (n * (n**2 - 1))
    rho_scipy, p_valor = stats.spearmanr(x, y)

    print("=" * 65)
    print(f"Spearman entre {nombre_x} y {nombre_y}")
    print("=" * 65)
    print(f"n = {n}")
    print("-" * 65)
    print(f"{'i':>3} {'x':>10} {'Rx':>8} {'y':>10} {'Ry':>8} {'d':>8} {'d2':>8}")
    print("-" * 65)
    for i in range(n):
        print(f"{i+1:>3} {x[i]:>10.2f} {rx[i]:>8.1f} {y[i]:>10.2f} {ry[i]:>8.1f} {d[i]:>8.1f} {d[i]**2:>8.1f}")
    print("-" * 65)
    print(f"Suma d2 = {sum_d2:.4f}")
    print(f"rho (formula suma d2) = {rho_formula:.4f}")
    print(f"rho (scipy)           = {rho_scipy:.4f}")
    print(f"p-valor               = {p_valor:.6e}")
    print("=" * 65)
    return rho_scipy, p_valor

## 3. Verificacion del Ejercicio 1 (cuaderno)

Datos: horas de estudio y calificacion.

In [ ]:
x1 = [1, 2, 4, 5, 7]
y1 = [55, 60, 75, 80, 90]

rho1, p1 = spearman_manual_detalle(x1, y1, "Horas de estudio", "Calificacion")

r1, _ = stats.pearsonr(x1, y1)
print(f"\nPearson r = {r1:.4f}  |  Spearman rho = {rho1:.4f}")

plt.figure(figsize=(8, 5))
sns.regplot(x=x1, y=y1, ci=None, scatter_kws={"s": 100},
            line_kws={"color": "red", "linewidth": 2})
plt.title(f"Ejercicio 1: rho = {rho1:.4f}")
plt.xlabel("Horas de estudio")
plt.ylabel("Calificacion")
plt.tight_layout()
plt.show()

## 4. Verificacion del Ejercicio 2 (relacion negativa)

In [ ]:
x2 = [40, 50, 60, 70, 80, 90]
y2 = [90, 75, 60, 50, 42, 35]

rho2, p2 = spearman_manual_detalle(x2, y2, "Velocidad", "Tiempo")

r2, _ = stats.pearsonr(x2, y2)
print(f"\nPearson r = {r2:.4f}  |  Spearman rho = {rho2:.4f}")

plt.figure(figsize=(8, 5))
sns.regplot(x=x2, y=y2, ci=None, scatter_kws={"s": 100, "color": "darkorange"},
            line_kws={"color": "red", "linewidth": 2})
plt.title(f"Ejercicio 2: rho = {rho2:.4f} (asociacion negativa)")
plt.xlabel("Velocidad (km/h)")
plt.ylabel("Tiempo (min)")
plt.tight_layout()
plt.show()

## 5. Efecto de un outlier (Ejercicio 3)

Comparamos Spearman y Pearson con y sin el outlier.

In [ ]:
x3_con = [8, 10, 12, 15, 18, 50]
y3_con = [1.2, 1.5, 1.8, 2.1, 2.4, 9.0]
x3_sin = [8, 10, 12, 15, 18]
y3_sin = [1.2, 1.5, 1.8, 2.1, 2.4]

print(">>> CON outlier:")
rho_con, _ = spearman_manual_detalle(x3_con, y3_con, "Ingreso", "Gasto")
r_con, _ = stats.pearsonr(x3_con, y3_con)

print("\n>>> SIN outlier:")
rho_sin, _ = spearman_manual_detalle(x3_sin, y3_sin, "Ingreso", "Gasto")
r_sin, _ = stats.pearsonr(x3_sin, y3_sin)

print("\n--- Resumen ---")
print(f"{'':20} {'Pearson r':>12} {'Spearman rho':>14}")
print(f"{'Con outlier':20} {r_con:>12.4f} {rho_con:>14.4f}")
print(f"{'Sin outlier':20} {r_sin:>12.4f} {rho_sin:>14.4f}")
print("Observa: Spearman suele cambiar menos que Pearson cuando hay un outlier extremo.")

## 6. Relacion monotonica no lineal

Ejemplo sintetico donde Pearson subestima y Spearman captura mejor la tendencia.

In [ ]:
np.random.seed(42)
n = 80
x_nl = np.linspace(0.5, 10, n)
y_nl = 3 * np.log(x_nl) + np.random.normal(0, 0.25, n)

r_nl, p_r = stats.pearsonr(x_nl, y_nl)
rho_nl, p_rho = stats.spearmanr(x_nl, y_nl)

print(f"Pearson  r   = {r_nl:.4f}  (p = {p_r:.2e})")
print(f"Spearman rho = {rho_nl:.4f}  (p = {p_rho:.2e})")

plt.figure(figsize=(8, 5))
plt.scatter(x_nl, y_nl, alpha=0.7)
plt.xlabel("X")
plt.ylabel("Y ~ log(X) + ruido")
plt.title(f"Relacion monotonica no lineal\nPearson r={r_nl:.3f}  |  Spearman rho={rho_nl:.3f}")
plt.tight_layout()
plt.show()

## 7. Dataset real: tips (seaborn)

In [ ]:
tips = sns.load_dataset("tips")
display(tips.head())

rho_tips, p_tips = stats.spearmanr(tips["total_bill"], tips["tip"])
r_tips, _ = stats.pearsonr(tips["total_bill"], tips["tip"])

print(f"total_bill vs tip:")
print(f"  Pearson  r   = {r_tips:.4f}")
print(f"  Spearman rho = {rho_tips:.4f}  (p = {p_tips:.2e})")

num = tips.select_dtypes(include="number")
corr_sp = num.corr(method="spearman")
print("\nMatriz de correlacion de Spearman:")
display(corr_sp.round(3))

plt.figure(figsize=(7, 5))
sns.heatmap(corr_sp, annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, fmt=".3f")
plt.title("Mapa de calor – Spearman (tips)")
plt.tight_layout()
plt.show()

## 8. Ejercicio 5 – Rankings de docentes

Los datos ya son rangos; calculamos Spearman entre los dos rankings.

In [ ]:
ranking_A = [1, 2, 3, 4, 5, 6]
ranking_B = [2, 1, 4, 3, 6, 5]

rho5, p5 = spearman_manual_detalle(ranking_A, ranking_B, "Ranking A", "Ranking B")
print("\nInterpretacion: valores altos de rho indican mayor concordancia entre docentes.")

## 9. Ejercicios adicionales (notebook)

### Ejercicio A
Con el dataset `tips`, elige otro par de variables numericas, calcula Pearson y Spearman,
y comenta si hay diferencias notables.

### Ejercicio B
Genera datos con un outlier artificial y muestra en una tabla como cambian r y rho
al incluir/excluir el outlier.

### Ejercicio C (reflexion en Markdown)
1. En que situacion del mundo real preferirias Spearman frente a Pearson?
2. Por que Spearman es mas robusto ante outliers?
3. Que limita a Spearman (que tipo de relacion no detecta bien)?


### Espacio para Ejercicio A

In [ ]:
# Tu codigo aqui


### Espacio para Ejercicio B

In [ ]:
# Tu codigo aqui


### Espacio para Ejercicio C

Escribe aqui tus respuestas:

1. 
2. 
3. 


## 10. Conclusiones

En esta practica hemos:

- Calculado Spearman con rangos y la formula de suma de d_i^2.
- Comparado sistematicamente con Pearson.
- Visto la mayor robustez ante outliers y la utilidad en relaciones monotonicas no lineales.
- Aplicado el coeficiente a rankings y a un dataset real.

**Buenas practicas**

| Aspecto | Buena practica |
|---------|----------------|
| Antes de decidir el coeficiente | Visualizar (scatter plot) |
| Outliers o no normalidad | Preferir Spearman (o Kendall) |
| Relacion lineal limpia | Pearson suele bastar |
| Reporte | rho, p-valor, n y motivo de la eleccion |

**Fin de la practica del subtema 2.1.2**
